# Lesson 8: Evals in production (CI gates, online evals, monitoring)

Companion to `curriculum/08-production.md`.

**Big idea:** offline evals guard the door, and online evals watch the house. You need both, and they feed each other.

Mock model only: free, offline, no API key. Run cells top to bottom.
**🧠 Check** = answer in your own words before moving on. **✍️ Your turn** = cells for you to write (ask Claude for a hint).

> Your thread through the course: an AI **validator** that reads an Issue from the GRC platform and checks
> its rating against the Incident & Issue Management Standard.

In [ ]:
import json, random, re
from collections import Counter, defaultdict

from evalkit import app, graders
from evalkit.stats import bootstrap_ci, mean, paired_bootstrap_delta, wilson, cohens_kappa
from nb_helpers import DEV, HOLDOUT, load, run, report, failures, mock_task

dev = load(DEV)
holdout = load(HOLDOUT)
print(f"dev={len(dev)} cases, holdout={len(holdout)} cases")

## 1. Offline evals in CI

| When | What | Gate? |
|---|---|---|
| Every PR touching prompts/models/AI code | smoke set (20–50 cases), code graders | **Block** on guardrails; warn on quality drops beyond noise |
| Nightly / pre-release | full set + holdout, multiple trials | release review |
| Model upgrade | full set side by side | explicit go/no-go |

**Gating rules that work:** guardrails are hard gates; quality gates need *paired CI excludes 0 AND drop >
X points*. Gating on raw deltas from small sets makes CI flaky, and people learn to ignore it.

Let's build a baseline and two candidate versions: one harmless, one with a policy regression.

In [ ]:
def harmless_task(text, seed=0):
    out = mock_task(text, seed=seed)
    out["reply"] = out["reply"].replace("Thanks for reaching out!", "Thanks for getting in touch!")
    return out

def risky_task(text, seed=0):
    out = mock_task(text, seed=seed)
    if out["category"] == "bug":
        out["reply"] += " We guarantee a fix by tomorrow."
    return out

baseline = run(dev, trials=3)
cand_ok = run(dev, trials=3, task=harmless_task)
cand_bad = run(dev, trials=3, task=risky_task)
for label, recs in [("baseline", baseline), ("harmless", cand_ok), ("risky", cand_bad)]:
    report(recs, trials=3, label=label)

### ✍️ Your turn: write the gate

Implement `gate(baseline, candidate)` that returns `(ok, reasons)`:
1. **Guardrail (hard):** if the `policy` pass rate drops at all, fail.
2. **Quality (soft-hard):** fail only if the overall pass rate drops by more than 5 points **and** the paired CI on the per-case delta excludes 0.

Helpers are provided. Lab 7 turns this into a real `ci_gate.py` script.

In [ ]:
def per_case(recs, grader=None):
    by = defaultdict(list)
    for r in recs:
        by[r["case_id"]].append(r["all_passed"] if grader is None
                                else next(g["passed"] for g in r["grades"] if g["name"] == grader))
    return {cid: mean(v) for cid, v in by.items()}

def gate(baseline, candidate):
    reasons = []
    # ✍️ Your code here. Use per_case(...) and paired_bootstrap_delta(...)
    return True, ["gate not implemented yet"]

for name, cand in [("harmless", cand_ok), ("risky", cand_bad)]:
    ok, reasons = gate(baseline, cand)
    print(f"{name:<9} {'PASS' if ok else 'BLOCK'}  {reasons}")

Expected: `harmless` → PASS, `risky` → BLOCK (policy).

### A GitHub Actions sketch

Earlier you asked about GitHub Actions. A PR gate could look like this (`ci_gate.py` is what you build in Lab 7):

```yaml
# .github/workflows/ai-eval.yml
name: ai-eval
on:
  pull_request:
    paths: ["labs/prompts/**", "labs/evalkit/**"]
jobs:
  eval:
    runs-on: ubuntu-latest
    defaults: { run: { working-directory: labs } }
    steps:
      - uses: actions/checkout@v4
      - uses: astral-sh/setup-uv@v5
      - run: uv run python run_eval.py --mock --no-judge --trials 3
      - run: uv run python ci_gate.py   # exits non-zero to block the merge
```
At work, the same shape applies in Jenkins or GitLab CI. The gate logic is what matters, not the CI vendor.

## 2. Online evals and monitoring

Production is where you find the cases you didn't think of. For the validator, useful **implicit signals**:
how often a human **overrides** its verdict, how often 2nd line **re-rates** an Issue it passed, and time to review.

Simulated weekly monitoring: override rate per week, plus input drift (share of a new Issue type).

In [ ]:
rng = random.Random(3)
print("week  issues  overrides  override_rate (95% CI)     new_type_share")
for week in range(1, 9):
    n = rng.randint(80, 120)
    drift = 0.02 + 0.04 * max(0, week - 4)          # a new Issue type starts appearing in week 5
    p_override = 0.06 + 0.5 * drift                 # validator is worse on the new type
    k = sum(rng.random() < p_override for _ in range(n))
    lo, hi = wilson(k, n)
    flag = "  <- investigate" if lo > 0.08 else ""
    print(f"{week:>4}  {n:>6}  {k:>9}  {k/n:5.1%}  [{lo:.1%}, {hi:.1%}]        {drift:5.0%}{flag}")

### 🧠 Check
The flag uses the **lower bound** of the CI (> 8%), not the raw rate. Why? And what's the very next thing
you'd do when it fires? (Hint: Lesson 3, the failure → test case loop.)

**Monitor, over time and per slice:** quality proxies (override rate, judge scores), guardrail violations,
errors, refusals and timeouts, cost per Issue, latency p50/p95, and **input drift**.

## 3. Closing the loop

```
production ──▶ monitors & sampling ──▶ failures found
     ▲                                      │
     │                                      ▼
ship change ◀── offline eval passes ◀── add as test cases
```

**Model upgrades** = dependency major-version bumps: full side-by-side eval, read the regressions, check
cost and latency, then decide.

---
## As a lead engineer…

- **Own the pipeline design:** which evals run where, who's paged on a guardrail breach, who triages online failures.
- **Define AI quality SLOs:** e.g. "override rate < X% per month; zero 2-level under-ratings in sampled review".
- **AI incident process:** if the validator wrongly passes a serious Issue, who rolls back the prompt, and how fast?

### 🧠 Discussion
In a regulated GRC setting, what evidence from this pipeline would you keep for audit (eval runs per
release? sampled human reviews? the gate decisions)? How long would you keep it?

**Recap**
- CI: guardrails block; quality gates must clear the noise (paired CI + threshold).
- Online: override and re-rating rates, sampled review, drift by slice, judged with CIs and not raw rates.
- Production failures flow back into the offline dataset. That loop is the operating rhythm.

**Next:** `/lab 7`, then `/lesson 9`.